# EgoBrain — EEG Feature Extraction

**Purpose:** Extract 1-second, 128-D time-indexed EEG features from the 32 physical Emotiv FLEX 2 EEG electrodes. The implementation streams one subject at a time to stay within Kaggle CPU/RAM/disk constraints.

## Reproducibility contract

This is the canonical raw-to-feature EEG preprocessing notebook. Run it once for all 40 subjects, or adapt the existing subject loop when resource limits require chunking.

### Inputs

- Gated `ut-vision/EgoBrain` dataset access.
- Kaggle secret `HF_TOKEN`.
- CPU Kaggle environment with sufficient temporary storage.

### Outputs

- `/kaggle/working/eeg_features/eeg_Pxxxx.pt`
- `/kaggle/working/eeg_features.zip`

---



In [ ]:
# ============================================================
# EgoBrain — Corrected EEG Stage 2 (EEG-only, time-indexed)
# ============================================================
#
# Purpose
# -------
# Build a scientifically clean EEG feature representation WITHOUT
# assuming that the number of EEG windows equals the number of
# visual feature rows.
#
# This script deliberately does NOT load features_Pxxxx.pt.
# Visual↔EEG pairing is a separate synchronization stage.
#
# Output per subject
# ------------------
#   eeg_Pxxxx.pt
#
# Dictionary:
#   {
#       "format_version": 2,
#       "modality": "eeg",
#       "subject_id": "P0001",
#       "feature_dim": 128,
#       "sampling_rate_hz": 256.0,
#       "window_seconds": 1.0,
#       "channel_names": [...32...],
#       "bands": {...},
#       "time_start_sec": Tensor[N],
#       "time_end_sec": Tensor[N],
#       "start_sample": Tensor[N],
#       "end_sample": Tensor[N],
#       "features": Tensor[N,128]
#   }
#
# Important:
#   The 32 physical EEG channels are selected explicitly by their
#   electrode names. EgoBrain EDFs also contain timestamps,
#   counters, marker signals, CQ/EQ quality channels, and CMS/DRL
#   reference signals; these are NOT EEG feature channels.
#
# ============================================================

import os
import sys
import gc
import csv
import json
import math
import time
import shutil
import hashlib
import subprocess
from pathlib import Path
from typing import List, Dict

# ============================================================
# 0. CONFIGURATION
# ============================================================

# Run ONE subject first.
SUBJECTS = [f"P{i:04d}" for i in range(1, 41)]

# After P0001 is fully validated, change to:
# SUBJECTS = [f"P{i:04d}" for i in range(1, 41)]

HF_REPO_ID = "ut-vision/EgoBrain"
HF_REPO_TYPE = "dataset"

WORKING_DIR = Path("/kaggle/working")
TEMP_DIR = Path("/kaggle/temp")

OUTPUT_DIR = WORKING_DIR / "eeg_features"
RAW_DIR = TEMP_DIR / "egobrain_eeg_raw"
HF_SCRATCH = TEMP_DIR / "hf_home"

MANIFEST_OUTPUT = WORKING_DIR / "eeg_stage2_manifest.json"
ERROR_OUTPUT = WORKING_DIR / "eeg_stage2_errors.jsonl"
CHECKSUM_OUTPUT = WORKING_DIR / "eeg_stage2_md5.txt"

EEG_SAMPLING_RATE = 256.0
WINDOW_SECONDS = 1.0
WINDOW_SAMPLES = 256

# Exact 32 physical EEG electrodes observed in P0001.
# Do not replace this with mne.pick_types(eeg=True).
EEG_CHANNEL_NAMES = [
    "Cz",
    "Fz",
    "Fp1",
    "F7",
    "F3",
    "FC1",
    "C3",
    "FC5",
    "FT9",
    "T7",
    "CP5",
    "CP1",
    "P3",
    "P7",
    "PO9",
    "O1",
    "Pz",
    "Oz",
    "O2",
    "PO10",
    "P8",
    "P4",
    "CP2",
    "CP6",
    "T8",
    "FT10",
    "FC6",
    "C4",
    "FC2",
    "F4",
    "F8",
    "Fp2",
]

assert len(EEG_CHANNEL_NAMES) == 32
assert len(set(EEG_CHANNEL_NAMES)) == 32

BANDS = {
    "theta": (4.0, 8.0),
    "alpha": (8.0, 12.0),
    "beta": (13.0, 30.0),
    "gamma": (30.0, 45.0),
}

FEATURE_DIM = 32 * len(BANDS)

WELCH_NPERSEG = 256
WELCH_NOVERLAP = 128

LOW_CUTOFF = 0.5
HIGH_CUTOFF = 45.0
NOTCH_FREQUENCY = 50.0
IIR_ORDER = 4

WINDOW_BATCH_SIZE = 128
HF_MAX_WORKERS = 1
MIN_FREE_WORKING_GB = 2.0

# ============================================================
# 1. DIRECTORIES / CACHE
# ============================================================

for path in [
    WORKING_DIR,
    TEMP_DIR,
    OUTPUT_DIR,
    RAW_DIR,
    HF_SCRATCH,
]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_SCRATCH)
os.environ["HF_HUB_CACHE"] = str(HF_SCRATCH / "hub")
os.environ["HF_XET_CACHE"] = str(HF_SCRATCH / "xet")
os.environ["HF_ASSETS_CACHE"] = str(HF_SCRATCH / "assets")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# ============================================================
# 2. PACKAGES
# ============================================================

def ensure_package(import_name: str, pip_name: str | None = None):
    try:
        __import__(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            pkg,
        ])

ensure_package("mne", "mne")
ensure_package("scipy", "scipy")
ensure_package("huggingface_hub", "huggingface_hub")

import numpy as np
import torch
from scipy import signal
import mne
from huggingface_hub import snapshot_download, login

mne.set_log_level("WARNING")

# ============================================================
# 3. UTILITIES
# ============================================================

def bytes_to_gb(value: int) -> float:
    return value / (1024 ** 3)


def free_gb(path: Path) -> float:
    return bytes_to_gb(shutil.disk_usage(path).free)


def cleanup_dir(path: Path):
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)


def cleanup_scratch():
    cleanup_dir(RAW_DIR)
    cleanup_dir(HF_SCRATCH)
    HF_SCRATCH.mkdir(parents=True, exist_ok=True)
    gc.collect()


def disk_report():
    w = shutil.disk_usage(WORKING_DIR)
    print(
        f"[DISK] /kaggle/working "
        f"free={bytes_to_gb(w.free):.2f} GB "
        f"used={bytes_to_gb(w.used):.2f} GB"
    )


def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        while True:
            block = f.read(16 * 1024 * 1024)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def log_error(subject_id: str, exc: Exception):
    record = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "subject_id": subject_id,
        "error_type": type(exc).__name__,
        "error": str(exc),
    }
    with ERROR_OUTPUT.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

# ============================================================
# 4. HF AUTH
# ============================================================

from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Kaggle secret HF_TOKEN is empty.")

login(token=HF_TOKEN, add_to_git_credential=False)
print("[AUTH] Hugging Face authentication OK.")

# ============================================================
# 5. DOWNLOAD ONE SUBJECT — EEG + MARKER + JSON
# ============================================================

def download_subject(subject_id: str):
    cleanup_scratch()

    if free_gb(WORKING_DIR) < MIN_FREE_WORKING_GB:
        raise RuntimeError(
            f"Only {free_gb(WORKING_DIR):.2f} GB free in /kaggle/working."
        )

    patterns = [
        f"{subject_id}/EmoTiv_FLEX_2_Gel/Phase_2/*.edf",
        f"{subject_id}/EmoTiv_FLEX_2_Gel/Phase_2/*_intervalMarker.csv",
        f"{subject_id}/EmoTiv_FLEX_2_Gel/Phase_2/*.json",
    ]

    ignore_patterns = [
        f"{subject_id}/EmoTiv_FLEX_2_Gel/Phase_2/*.md.edf",
        f"{subject_id}/EmoTiv_FLEX_2_Gel/Phase_2/*_survey.csv",
    ]

    print(f"[DOWNLOAD] {subject_id}")

    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        allow_patterns=patterns,
        ignore_patterns=ignore_patterns,
        local_dir=str(RAW_DIR),
        token=HF_TOKEN,
        max_workers=HF_MAX_WORKERS,
        force_download=False,
    )

# ============================================================
# 6. LOCATE PRIMARY EDF / MARKER
# ============================================================

def phase_dir(subject_id: str) -> Path:
    return (
        RAW_DIR
        / subject_id
        / "EmoTiv_FLEX_2_Gel"
        / "Phase_2"
    )


def find_primary_edf(subject_id: str) -> Path:
    phase = phase_dir(subject_id)
    edfs = [
        p for p in phase.glob("*.edf")
        if not p.name.lower().endswith(".md.edf")
    ]

    if len(edfs) != 1:
        raise RuntimeError(
            f"Expected exactly one primary EDF for {subject_id}; "
            f"found {len(edfs)}: {[p.name for p in edfs]}"
        )

    return edfs[0]


def find_marker(edf_path: Path) -> Path:
    exact = edf_path.with_name(
        edf_path.stem + "_intervalMarker.csv"
    )

    if exact.exists():
        return exact

    markers = list(edf_path.parent.glob("*_intervalMarker.csv"))

    if len(markers) == 1:
        return markers[0]

    raise RuntimeError(
        f"Could not uniquely identify interval marker for {edf_path.name}."
    )

# ============================================================
# 7. LOAD EXACT 32 EEG CHANNELS
# ============================================================

def load_eeg_32(edf_path: Path):
    print(f"[EEG] Reading header: {edf_path.name}")

    # Do not preload auxiliary channels.
    raw = mne.io.read_raw_edf(
        edf_path,
        preload=False,
        verbose=False,
    )

    sfreq = float(raw.info["sfreq"])
    duration = raw.n_times / sfreq

    print(
        f"[EEG] Raw channels={len(raw.ch_names)} "
        f"Fs={sfreq:.3f} Hz "
        f"Duration={duration:.3f}s"
    )

    if not math.isclose(
        sfreq,
        EEG_SAMPLING_RATE,
        abs_tol=0.05,
    ):
        raise RuntimeError(
            f"Expected {EEG_SAMPLING_RATE} Hz, found {sfreq} Hz."
        )

    missing = [
        name
        for name in EEG_CHANNEL_NAMES
        if name not in raw.ch_names
    ]

    if missing:
        raise RuntimeError(
            "Required physical EEG electrodes are missing: "
            + ", ".join(missing)
        )

    # Ordered selection by exact physical electrode names.
    picks = mne.pick_channels(
        raw.ch_names,
        include=EEG_CHANNEL_NAMES,
        ordered=True,
    )

    if len(picks) != 32:
        raise RuntimeError(
            f"Expected 32 selected channels, found {len(picks)}."
        )

    raw.pick(picks)

    if raw.ch_names != EEG_CHANNEL_NAMES:
        raise RuntimeError(
            "Channel order mismatch after selection.\n"
            f"Expected={EEG_CHANNEL_NAMES}\n"
            f"Found={raw.ch_names}"
        )

    # Load only the 32 physical EEG channels.
    raw.load_data()

    print("[EEG] Selected 32 physical EEG electrodes:")
    print("[EEG] " + ", ".join(raw.ch_names))

    return raw

# ============================================================
# 8. FILTER
# ============================================================

def filter_eeg(raw):
    print("[FILTER] 50 Hz notch first...")

    raw.notch_filter(
        freqs=[NOTCH_FREQUENCY],
        method="iir",
        iir_params={
            "order": IIR_ORDER,
            "ftype": "butter",
        },
        phase="zero",
        n_jobs=1,
        verbose=False,
    )

    print(
        f"[FILTER] {LOW_CUTOFF}-{HIGH_CUTOFF} Hz bandpass..."
    )

    raw.filter(
        l_freq=LOW_CUTOFF,
        h_freq=HIGH_CUTOFF,
        method="iir",
        iir_params={
            "order": IIR_ORDER,
            "ftype": "butter",
        },
        phase="zero",
        n_jobs=1,
        verbose=False,
    )

    return raw

# ============================================================
# 9. WELCH BAND POWER
# ============================================================

def band_power(freqs, psd, low, high):
    mask = (freqs >= low) & (freqs < high)

    if high == 45.0:
        mask = (freqs >= low) & (freqs <= high)

    if not np.any(mask):
        raise RuntimeError(
            f"No Welch frequency bins for {low}-{high} Hz."
        )

    f = freqs[mask]
    p = psd[..., mask]

    if len(f) >= 2:
        if hasattr(np, "trapezoid"):
            return np.trapezoid(p, f, axis=-1)
        return np.trapz(p, f, axis=-1)

    return p[..., 0]


def features_from_chunk(eeg_chunk: np.ndarray) -> np.ndarray:
    # eeg_chunk = [32, N*256]
    channels, samples = eeg_chunk.shape

    if channels != 32:
        raise RuntimeError(
            f"Expected [32,T], got {eeg_chunk.shape}"
        )

    if samples % WINDOW_SAMPLES != 0:
        raise RuntimeError(
            f"Chunk samples {samples} not divisible by {WINDOW_SAMPLES}."
        )

    n_windows = samples // WINDOW_SAMPLES

    windows = (
        eeg_chunk
        .reshape(32, n_windows, WINDOW_SAMPLES)
        .transpose(1, 0, 2)
        .astype(np.float32, copy=False)
    )

    if not np.isfinite(windows).all():
        raise RuntimeError("NaN/Inf detected in EEG chunk.")

    freqs, psd = signal.welch(
        windows,
        fs=EEG_SAMPLING_RATE,
        window="hann",
        nperseg=WELCH_NPERSEG,
        noverlap=WELCH_NOVERLAP,
        detrend="constant",
        return_onesided=True,
        scaling="density",
        axis=-1,
    )

    if not np.isfinite(psd).all():
        raise RuntimeError("NaN/Inf detected in Welch PSD.")

    band_vectors = []

    for low, high in BANDS.values():
        power = band_power(
            freqs,
            psd,
            low,
            high,
        )
        band_vectors.append(
            power.astype(np.float32, copy=False)
        )

    # [N,32] x 4 -> [N,32,4] -> [N,128]
    features = np.stack(
        band_vectors,
        axis=-1,
    ).reshape(
        n_windows,
        FEATURE_DIM,
    )

    if not np.isfinite(features).all():
        raise RuntimeError("NaN/Inf in band-power features.")

    return features.astype(np.float32, copy=False)

# ============================================================
# 10. EXTRACT THE ENTIRE EEG CLOCK IN 1-SECOND WINDOWS
# ============================================================

def extract_eeg_timeline(raw):
    n_complete_windows = raw.n_times // WINDOW_SAMPLES

    if n_complete_windows <= 0:
        raise RuntimeError("EEG is shorter than one complete second.")

    total_samples = n_complete_windows * WINDOW_SAMPLES

    print(
        f"[EEG] Complete 1-sec windows={n_complete_windows:,}"
    )
    print(
        f"[EEG] Samples used={total_samples:,} / {raw.n_times:,}"
    )

    chunks = []
    completed = 0

    while completed < n_complete_windows:
        n_win = min(
            WINDOW_BATCH_SIZE,
            n_complete_windows - completed,
        )

        start = completed * WINDOW_SAMPLES
        stop = start + n_win * WINDOW_SAMPLES

        chunk = raw.get_data(
            start=start,
            stop=stop,
        )

        if chunk.shape != (
            32,
            n_win * WINDOW_SAMPLES,
        ):
            raise RuntimeError(
                f"Unexpected EEG chunk shape={chunk.shape}"
            )

        features = features_from_chunk(chunk)
        chunks.append(torch.from_numpy(features))

        completed += n_win

        if (
            completed % (WINDOW_BATCH_SIZE * 10) == 0
            or completed == n_complete_windows
        ):
            print(
                f"[FEATURES] {completed:,}/{n_complete_windows:,}"
            )

        del chunk
        del features
        gc.collect()

    feature_tensor = torch.cat(chunks, dim=0).float()
    chunks.clear()

    if feature_tensor.shape != (
        n_complete_windows,
        FEATURE_DIM,
    ):
        raise RuntimeError(
            f"Unexpected final feature shape="
            f"{tuple(feature_tensor.shape)}"
        )

    if not torch.isfinite(feature_tensor).all():
        raise RuntimeError("Final EEG feature tensor contains NaN/Inf.")

    time_start = torch.arange(
        n_complete_windows,
        dtype=torch.float64,
    )

    time_end = time_start + WINDOW_SECONDS

    start_sample = (
        torch.arange(
            n_complete_windows,
            dtype=torch.int64,
        ) * WINDOW_SAMPLES
    )

    end_sample = start_sample + WINDOW_SAMPLES

    return {
        "features": feature_tensor,
        "time_start_sec": time_start,
        "time_end_sec": time_end,
        "start_sample": start_sample,
        "end_sample": end_sample,
    }

# ============================================================
# 11. COPY MARKER TABLE INTO PROVENANCE
# ============================================================

def read_marker_rows(marker_path: Path):
    with marker_path.open(
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline="",
    ) as f:
        reader = csv.DictReader(f)
        return list(reader)

# ============================================================
# 12. PROCESS ONE SUBJECT
# ============================================================

def process_subject(subject_id: str):
    output_path = OUTPUT_DIR / f"eeg_{subject_id}.pt"

    if output_path.exists():
        try:
            existing = torch.load(
                output_path,
                map_location="cpu",
                weights_only=True,
            )
            if (
                isinstance(existing, dict)
                and existing.get("modality") == "eeg"
                and int(existing.get("feature_dim", -1)) == FEATURE_DIM
                and torch.is_tensor(existing.get("features"))
                and existing["features"].ndim == 2
                and existing["features"].shape[1] == FEATURE_DIM
            ):
                print(
                    f"[{subject_id}] Valid EEG-only checkpoint -> SKIP"
                )
                return {
                    "subject_id": subject_id,
                    "status": "skipped_existing",
                    "shape": list(existing["features"].shape),
                }
        except Exception:
            pass

    print("\n" + "=" * 80)
    print(f"SUBJECT {subject_id}")
    print("=" * 80)

    download_subject(subject_id)

    edf_path = find_primary_edf(subject_id)
    marker_path = find_marker(edf_path)

    raw = None

    try:
        raw = load_eeg_32(edf_path)

        raw = filter_eeg(raw)

        timeline = extract_eeg_timeline(raw)

        marker_rows = read_marker_rows(marker_path)

        dataset = {
            "format_version": 2,
            "modality": "eeg",
            "subject_id": subject_id,
            "feature_dim": FEATURE_DIM,
            "sampling_rate_hz": EEG_SAMPLING_RATE,
            "window_seconds": WINDOW_SECONDS,
            "window_samples": WINDOW_SAMPLES,
            "channel_names": EEG_CHANNEL_NAMES,
            "bands": BANDS,
            "welch": {
                "nperseg": WELCH_NPERSEG,
                "noverlap": WELCH_NOVERLAP,
            },
            "filter": {
                "notch_hz": NOTCH_FREQUENCY,
                "bandpass_low_hz": LOW_CUTOFF,
                "bandpass_high_hz": HIGH_CUTOFF,
                "method": "zero-phase Butterworth IIR",
                "order": IIR_ORDER,
            },
            "source": {
                "edf": edf_path.name,
                "marker_csv": marker_path.name,
                "marker_rows": marker_rows,
            },
            "features": timeline["features"],
            "time_start_sec": timeline["time_start_sec"],
            "time_end_sec": timeline["time_end_sec"],
            "start_sample": timeline["start_sample"],
            "end_sample": timeline["end_sample"],
        }

        temp = output_path.with_suffix(".pt.tmp")
        torch.save(dataset, temp)

        verify = torch.load(
            temp,
            map_location="cpu",
            weights_only=True,
        )

        if (
            not isinstance(verify, dict)
            or verify.get("modality") != "eeg"
            or verify.get("feature_dim") != FEATURE_DIM
            or not torch.is_tensor(verify.get("features"))
            or verify["features"].shape[1] != FEATURE_DIM
            or not torch.isfinite(verify["features"]).all()
        ):
            raise RuntimeError("Post-save verification failed.")

        temp.replace(output_path)

        print("\n[SUBJECT COMPLETE]")
        print(f"Subject          : {subject_id}")
        print(f"EEG features     : {tuple(timeline['features'].shape)}")
        print(f"Channel count    : {len(EEG_CHANNEL_NAMES)}")
        print(f"Clock covered    : {timeline['time_start_sec'][0].item():.3f} -> "
              f"{timeline['time_end_sec'][-1].item():.3f} s")
        print(f"Output           : {output_path}")

        return {
            "subject_id": subject_id,
            "status": "success",
            "shape": list(timeline["features"].shape),
            "clock_start_sec": float(timeline["time_start_sec"][0]),
            "clock_end_sec": float(timeline["time_end_sec"][-1]),
            "output_bytes": output_path.stat().st_size,
        }

    finally:
        raw = None
        gc.collect()
        cleanup_scratch()
        disk_report()

# ============================================================
# 13. MAIN
# ============================================================

def main():
    start_time = time.time()

    manifest = {
        "format_version": 2,
        "pipeline": "eeg_only_time_indexed",
        "feature_dim": FEATURE_DIM,
        "sampling_rate_hz": EEG_SAMPLING_RATE,
        "window_seconds": WINDOW_SECONDS,
        "channels": EEG_CHANNEL_NAMES,
        "bands": BANDS,
        "subjects": {},
    }

    failures = []

    for i, subject_id in enumerate(SUBJECTS, start=1):
        print("\n" + "#" * 80)
        print(f"SUBJECT {i}/{len(SUBJECTS)}: {subject_id}")
        print("#" * 80)

        try:
            result = process_subject(subject_id)
            manifest["subjects"][subject_id] = result
        except Exception as exc:
            print(
                f"[SUBJECT FAILURE] {subject_id}: "
                f"{type(exc).__name__}: {exc}"
            )
            log_error(subject_id, exc)
            failures.append(subject_id)
            manifest["subjects"][subject_id] = {
                "subject_id": subject_id,
                "status": "failed",
                "error_type": type(exc).__name__,
                "error": str(exc),
            }

        MANIFEST_OUTPUT.write_text(
            json.dumps(manifest, indent=2),
            encoding="utf-8",
        )

    # Checksums
    with CHECKSUM_OUTPUT.open("w", encoding="utf-8") as f:
        for subject_id in SUBJECTS:
            p = OUTPUT_DIR / f"eeg_{subject_id}.pt"
            if p.exists():
                f.write(f"{md5_file(p)}  {p.name}\n")

    completed = [
        sid
        for sid in SUBJECTS
        if (OUTPUT_DIR / f"eeg_{sid}.pt").exists()
    ]

    manifest["completed_subjects"] = completed
    manifest["failed_subjects"] = failures
    manifest["elapsed_seconds"] = time.time() - start_time

    MANIFEST_OUTPUT.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    print("\n" + "=" * 80)
    print("FINAL EEG-ONLY STAGE-2 REPORT")
    print("=" * 80)
    print(f"Completed: {len(completed)}/{len(SUBJECTS)}")
    print(f"Failed   : {len(failures)}/{len(SUBJECTS)}")
    print(f"Manifest : {MANIFEST_OUTPUT}")
    print(f"Errors   : {ERROR_OUTPUT}")
    print(f"MD5      : {CHECKSUM_OUTPUT}")
    disk_report()


if __name__ == "__main__":
    main()


In [ ]:
import os
import shutil
from IPython.display import FileLink

# 1. Create a zip archive of the /kaggle/working directory
# This saves 'output.zip' directly inside /kaggle/working
shutil.make_archive('output', 'zip', '/kaggle/working')

# 2. Generate a clickable download link in your notebook
FileLink('output.zip')
